In [29]:
spark.stop()

In [9]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Understand DAG Plan")
    .master("local[*]")
    .getOrCreate()
)

spark


In [10]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [11]:
# Check default Parallism

spark.sparkContext.defaultParallelism

4

In [12]:
# create dataframes

df_1 = spark.range(4, 200, 2)
df_2 = spark.range(2, 200, 4)

In [14]:
df_2.rdd.getNumPartitions()

4

In [15]:
# Re-partition data

df_3 = df_1.repartition(2)
df_4 = df_2.repartition(3)

In [16]:
df_4.rdd.getNumPartitions()

3

In [ ]:
df_4.show()

In [ ]:
# join the dataframe

df_joined = df_3.join(df_4, on="id", how="inner")

In [22]:
df_joined.rdd.getNumPartitions()

200

In [ ]:
df_sum = df_joined.selectExpr("sum(id) as total_sum")

In [23]:
df_sum.rdd.getNumPartitions()

1

In [20]:
df_sum.show()

+---------+
|total_sum|
+---------+
|     4998|
+---------+



In [21]:
df_sum.explain()

== Physical Plan ==
*(6) HashAggregate(keys=[], functions=[sum(id#0L)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#189]
   +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#0L)])
      +- *(5) Project [id#0L]
         +- *(5) SortMergeJoin [id#0L], [id#2L], Inner
            :- *(2) Sort [id#0L ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(id#0L, 200), ENSURE_REQUIREMENTS, [id=#173]
            :     +- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [id=#172]
            :        +- *(1) Range (4, 200, step=2, splits=4)
            +- *(4) Sort [id#2L ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(id#2L, 200), ENSURE_REQUIREMENTS, [id=#180]
                  +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [id=#179]
                     +- *(3) Range (2, 200, step=4, splits=4)




In [ ]:
df_union = df_sum.union(df_4)

In [26]:
df_union.show()

+---------+
|total_sum|
+---------+
|     4998|
|       14|
|       10|
|       42|
|       18|
|       74|
|       54|
|       70|
|       78|
|      122|
|      134|
|      142|
|      138|
|      170|
|      190|
|      178|
|      166|
|      174|
|       26|
|       46|
+---------+
only showing top 20 rows



In [27]:
df_union.explain()

== Physical Plan ==
Union
:- *(6) HashAggregate(keys=[], functions=[sum(id#0L)])
:  +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [id=#484]
:     +- *(5) HashAggregate(keys=[], functions=[partial_sum(id#0L)])
:        +- *(5) Project [id#0L]
:           +- *(5) SortMergeJoin [id#0L], [id#2L], Inner
:              :- *(2) Sort [id#0L ASC NULLS FIRST], false, 0
:              :  +- Exchange hashpartitioning(id#0L, 200), ENSURE_REQUIREMENTS, [id=#468]
:              :     +- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [id=#467]
:              :        +- *(1) Range (4, 200, step=2, splits=4)
:              +- *(4) Sort [id#2L ASC NULLS FIRST], false, 0
:                 +- Exchange hashpartitioning(id#2L, 200), ENSURE_REQUIREMENTS, [id=#475]
:                    +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [id=#474]
:                       +- *(3) Range (2, 200, step=4, splits=4)
+- ReusedExchange [id#27L], Exchange RoundRobinPartitioning(3), REPARTITION_BY_N

In [28]:
df_1.rdd

MapPartitionsRDD[127] at javaToPython at NativeMethodAccessorImpl.java:0